In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json

ModuleNotFoundError: No module named 'matplotlib'

In [ ]:
def analyze_dynamic_bias(csv_path):
    # 1. Load Data
    df = pd.read_csv(csv_path)

    # 2. Parse Demographics from group_label (e.g., 'Black_Female_Mechanic')
    df[['Race', 'Gender', 'Occupation']] = df['group_label'].str.split('_', n=2, expand=True)
    df['Race_Gender'] = df['Race'] + ' ' + df['Gender']

    # ==========================================
    # AGGREGATION 1: Goal Completion Rate (GCR)
    # ==========================================
    df['GCR_diff'] = df['explicit_GCR'] - df['implicit_GCR']
    gcr_by_race_gender = df.groupby('Race_Gender')[['implicit_GCR', 'explicit_GCR', 'GCR_diff']].mean().sort_values('GCR_diff')
    
    print("--- 1. GCR Differences (Explicit - Implicit) ---")
    print(gcr_by_race_gender)

    # ==========================================
    # AGGREGATION 2: Semantic Steering
    # ==========================================
    df['Steering_diff'] = df['explicit_Steering'] - df['implicit_Steering']
    steering_by_race_gender = df.groupby('Race_Gender')[['implicit_Steering', 'explicit_Steering', 'Steering_diff']].mean().sort_values('Steering_diff', ascending=False)
    
    print("\n--- 2. Semantic Steering Shifts (Explicit - Implicit) ---")
    print(steering_by_race_gender)

    # ==========================================
    # AGGREGATION 3: Turn-by-Turn Trajectory
    # ==========================================
    def parse_traj(x):
        try:
            return np.array(json.loads(x))
        except:
            return np.array([])

    df['imp_traj'] = df['implicit_Steering_Trajectory'].apply(parse_traj)
    df['exp_traj'] = df['explicit_Steering_Trajectory'].apply(parse_traj)

    max_turns = 5
    def pad_traj(arr, length=max_turns):
        if len(arr) == 0: return np.full(length, np.nan)
        if len(arr) < length: return np.pad(arr, (0, length - len(arr)), constant_values=np.nan)
        return arr[:length]

    # Aggregate trajectories across all transcripts
    imp_matrix = np.vstack(df['imp_traj'].apply(pad_traj).values)
    exp_matrix = np.vstack(df['exp_traj'].apply(pad_traj).values)

    imp_mean_traj = np.nanmean(imp_matrix, axis=0)
    exp_mean_traj = np.nanmean(exp_matrix, axis=0)

    # ==========================================
    # GRAPH GENERATION
    # ==========================================
    sns.set_theme(style="whitegrid")

    # Plot 1: Goal Completion Rate Drop
    fig, ax = plt.subplots(figsize=(12, 6))
    gcr_by_race_gender[['implicit_GCR', 'explicit_GCR']].plot(kind='bar', ax=ax, color=['#8cb3d9', '#d95f02'])
    ax.set_title('Allocational Harm: Goal Completion Rate Drop (Implicit vs Explicit)', fontsize=14)
    ax.set_ylabel('Average Goal Completion Rate (GCR)')
    ax.set_xlabel('Intersectional Identity')
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
    ax.legend(['Implicit Identity', 'Explicit Identity'])
    plt.tight_layout()
    plt.savefig('1_GCR_Implicit_vs_Explicit.png', dpi=300)
    plt.close()

    # Plot 2: Semantic Steering Shift
    fig, ax = plt.subplots(figsize=(12, 6))
    steering_by_race_gender[['implicit_Steering', 'explicit_Steering']].plot(kind='bar', ax=ax, color=['#8cb3d9', '#d95f02'])
    ax.set_title('Representational Harm: Semantic Stereotyping (Implicit vs Explicit)', fontsize=14)
    ax.set_ylabel('Average Semantic Steering Score')
    ax.set_xlabel('Intersectional Identity')
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
    ax.legend(['Implicit Identity', 'Explicit Identity'])
    plt.tight_layout()
    plt.savefig('2_Steering_Implicit_vs_Explicit.png', dpi=300)
    plt.close()

    # Plot 3: Turn-by-Turn Trajectory (Overall)
    plt.figure(figsize=(9, 6))
    turns = np.arange(1, max_turns + 1)
    plt.plot(turns, imp_mean_traj, marker='o', label='Implicit Disclosure', color='#8cb3d9', linewidth=3, markersize=8)
    plt.plot(turns, exp_mean_traj, marker='s', label='Explicit Disclosure', color='#d95f02', linewidth=3, markersize=8)
    
    plt.title('Dynamic Bias Confirmed: Conversational Drift Over Time', fontsize=15, fontweight='bold')
    plt.xlabel('Conversation Turn', fontsize=12)
    plt.ylabel('Semantic Steering (Alignment to Stereotype)', fontsize=12)
    plt.xticks(turns)
    plt.legend(fontsize=12)
    plt.tight_layout()
    plt.savefig('3_Mean_Trajectory_Overall.png', dpi=300)
    plt.close()

    # Plot 4: Worst-Case Trajectory (Native American Female)
    worst_group = "Native American Female"
    worst_group_df = df[df['Race_Gender'] == worst_group]

    wg_imp_mat = np.vstack(worst_group_df['imp_traj'].apply(pad_traj).values)
    wg_exp_mat = np.vstack(worst_group_df['exp_traj'].apply(pad_traj).values)

    plt.figure(figsize=(9, 6))
    plt.plot(turns, np.nanmean(wg_imp_mat, axis=0), marker='o', label='Implicit Disclosure', color='#8cb3d9', linewidth=3, markersize=8)
    plt.plot(turns, np.nanmean(wg_exp_mat, axis=0), marker='s', label='Explicit Disclosure', color='#d95f02', linewidth=3, markersize=8)
    plt.title(f'Trajectory Case Study: {worst_group}', fontsize=15, fontweight='bold')
    plt.xlabel('Conversation Turn', fontsize=12)
    plt.ylabel('Semantic Steering (Alignment to Stereotype)', fontsize=12)
    plt.xticks(turns)
    plt.legend(fontsize=12)
    plt.tight_layout()
    plt.savefig('4_Trajectory_Worst_Group.png', dpi=300)
    plt.close()

    print("\nVisualizations saved successfully!")

In [ ]:
analyze_dynamic_bias('../Llama-3.1-70B-Instruct-AWQ-INT4/dynamic_bias_results.csv')